# Ornn GPU compute cost → `ornn_gpu_prices`

Daily settled **$/hr index** per GPU (Ornn OCPI, free tier — no key). Two inputs, upserted into one
append-style table keyed on `(date, gpu_name)`:

1. **Seed** — the 3-month history CSVs downloaded from Ornn (`databricks/compute/ornn-public-index-*.csv`).
2. **Daily** — the current settled value pulled live (keyless) each run.

Idempotent: re-running preserves accumulated days and updates today's value — safe as a daily run-all.
Free GPUs: H100 SXM, H200, B200, A100 SXM4, RTX 5090 (RTX PRO 6000 WS is paid — skipped).

In [ ]:
CATALOG, SCHEMA = "fso_market_intelligence", "frontier_labs"
TABLE = f"{CATALOG}.{SCHEMA}.ornn_gpu_prices"
BASE = "https://api.ornnai.com"

from datetime import datetime, timezone
NOW = datetime.now(timezone.utc).isoformat()
TODAY = datetime.now(timezone.utc).date().isoformat()
print("run:", NOW)

In [ ]:
# ── 1. seed rows from the downloaded 3-month CSVs ──
import subprocess, glob, pandas as pd
hits = subprocess.run(["find", "/Workspace", "-path", "*/databricks/compute/ornn-public-index-*.csv"],
                      capture_output=True, text=True).stdout.strip().splitlines()
print(f"seed CSVs found: {len(hits)}")
seed = []
for f in hits:
    d = pd.read_csv(f)
    gpu_col, ts_col, val_col = d.columns[0], d.columns[1], d.columns[2]   # Index, Timestamp (UTC), {gpu} index
    for _, r in d.iterrows():
        if pd.isna(r[val_col]):
            continue
        seed.append({"date": str(r[ts_col])[:10], "gpu_name": str(r[gpu_col]).strip(),
                     "index_value_usd_hr": float(r[val_col]), "source": "ornn_history",
                     "source_updated_at": str(r[ts_col]), "captured_at": NOW})
print("seed rows:", len(seed))

In [ ]:
# ── 2. today's live pull (keyless) ──
import requests, urllib.parse
gpus = [x["gpu_name"] for x in requests.get(f"{BASE}/api/gpu-types", timeout=30).json()["data"]]
today_rows = []
for g in gpus:
    r = requests.get(f"{BASE}/api/gpu/{urllib.parse.quote(g)}", timeout=30)
    if r.status_code != 200:
        print(f"  [skip] {g}: HTTP {r.status_code}"); continue
    dd = r.json()["data"]
    today_rows.append({"date": TODAY, "gpu_name": g, "index_value_usd_hr": float(dd["index_value"]),
                       "source": "ornn_daily", "source_updated_at": dd.get("last_updated"), "captured_at": NOW})
    print(f"  {g:16} ${dd['index_value']}/hr")
print("today rows:", len(today_rows))

In [ ]:
# ── 3. upsert into ornn_gpu_prices (preserve prior days; dedup by date+gpu, keep newest capture) ──
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("date", StringType()), StructField("gpu_name", StringType()),
    StructField("index_value_usd_hr", DoubleType()), StructField("source", StringType()),
    StructField("source_updated_at", StringType()), StructField("captured_at", StringType()),
])
new_df = spark.createDataFrame(seed + today_rows, schema=schema)

combined = (spark.table(TABLE).select(new_df.columns).unionByName(new_df)
            if spark.catalog.tableExists(TABLE) else new_df)
w = Window.partitionBy("date", "gpu_name").orderBy(F.col("captured_at").desc())
out = (combined.withColumn("_r", F.row_number().over(w)).filter("_r = 1").drop("_r")
       .withColumn("date", F.to_date("date")).withColumn("captured_at", F.to_timestamp("captured_at")))
(out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE))

n = spark.table(TABLE)
print("ornn_gpu_prices rows:", n.count(),
      "| days:", n.select("date").distinct().count(),
      "| GPUs:", sorted(x[0] for x in n.select("gpu_name").distinct().collect()))
display(n.orderBy(F.col("date").desc(), "gpu_name").limit(12))

## Scheduling
One **daily** Job (this notebook, **Source = Git provider / branch `main`**, serverless). Keyless — no
secret needed. The seed cell is idempotent (re-upserts the same history), so a daily run-all is safe.